In [ ]:
import time

import finesse
import ifotools.virgo.model_info as model_info_virgo
import ifotools.virgo.plotting as plotting_virgo
import ifotools.virgo.pretuning as pretuning_virgo
import matplotlib.pyplot as plt
from ifotools.finesse import model_info, plotting, pretuning

finesse.init_plotting()
import numpy as np

from finesse.utilities.tables import Table
import finesse.analysis.actions as fac

import matplotlib

In [ ]:
def cart2pol(x, y):
    """Convert from Cartesian to polar coordinates.

    Example
    -------
    >>> theta, radius = pol2cart(x, y)
    """
    radius = np.hypot(x, y)
    theta = np.arctan2(y, x)
    return theta, radius


def compass(u, v, arrowprops=None):
    """
    Compass draws a graph that displays the vectors with
    components `u` and `v` as arrows from the origin.

    Examples
    --------
    >>> import numpy as np
    >>> u = [+0, +0.5, -0.50, -0.90]
    >>> v = [+1, +0.5, -0.45, +0.85]
    >>> compass(u, v)
    """

    angles, radii = cart2pol(u, v)
    fig, ax = plt.subplots(subplot_kw=dict(polar=True))

    kw = dict(arrowstyle="->")
    if arrowprops:
        kw.update(arrowprops)
    [
        ax.annotate("", xy=(angle, radius), xytext=(0, 0), arrowprops=kw)
        for angle, radius in zip(angles, radii)
    ]

    ax.set_ylim(0, np.max(radii))
    return fig, ax


def get_QNLS(model, axis=[10, 1000, 500]):
    kat = model.deepcopy()
    kat.parse(
        """
        # Differentially modulate the arm lengths
        fsig(1)
        sgen darmx sNarm.h
        sgen darmy sWarm.h phase=180

        # Output the full quantum noise limited sensitivity
        qnoised NSR_with_RP SRM1.p2.o nsr=True

        # Output just the shot noise limited sensitivity
        qshot NSR_without_RP SRM1.p2.o nsr=True
        """
    )

    return kat.run(f'xaxis(darmx.f, "log", {axis[0]}, {axis[1]}, {axis[2]})')


def dof_plot_rf(model, dof, detector, axis=[-1, 1, 1000], xscale=1, logy=True):
    kat = model.deepcopy()
    axis = np.array(axis, dtype=np.float64)
    axis[:2] *= xscale
    out = kat.run(fac.Xaxis(eval("kat." + dof + ".DC"), "lin", *axis, relative=True))
    fig, ax = plt.subplots(1, 1, figsize=(5, 3))

    ax.plot(out.x[0], out[detector + "_I"], label=detector + "_I", linestyle="-")
    ax.plot(out.x[0], out[detector + "_Q"], label=detector + "_Q", linestyle="--")
    ax.set_title(dof + " DC scan")
    ax.set_xlabel(dof + " DC")
    ax.set_ylabel(r"Amplitude $[\sqrt{W}]$")
    ax.legend(ncol=2)
    ax.grid(True)
    fig.set_tight_layout(True)
    return out


def get_dxdy(kat, bp, port, f="0"):
    # this function takes a kat file (should already contain a misalignment and be locked afterwards),
    # a beam tracing output (bp = some_kat.beam_trace()) and a port attached to a mirror where the de-
    # centering is computed.
    #
    # It attaches the necessary ADs to a temporary copy of the kat file and runs a 1-point simulation,
    # then returns the dx/dy decentering.

    if f not in ["0", "eom1.f", "eom2.f", "eom3.f"]:
        raise ValueError(
            f"Invalid value for f: {f}. Expected one of ['0', 'eom1.f', 'eom2.f', 'eom3.f']."
        )

    mir = port.split(".")[0]
    tmp_kat = kat.deepcopy()

    tmp_kat.parse(f"""
    ad {mir}_00 {port} {f} 0 0
    ad {mir}_10 {port} {f} 1 0
    ad {mir}_01 {port} {f} 0 1
    """)

    tmp_kat.parse("noxaxis(pre_step=none, post_step=none, name='noxaxis')")
    out = tmp_kat.run()

    dx = (
        abs(out[f"{mir}_10"])
        / abs(out[f"{mir}_00"])
        * bp[f"{port}"][0].w
        * 1e6
        * np.cos(np.angle(out[f"{mir}_10"]) - np.angle(out[f"{mir}_00"]))
    )
    dy = (
        abs(out[f"{mir}_01"])
        / abs(out[f"{mir}_00"])
        * bp[f"{port}"][1].w
        * 1e6
        * np.cos(np.angle(out[f"{mir}_01"]) - np.angle(out[f"{mir}_00"]))
    )

    return dx, dy


def make_SoS_QDs(Bx, kat, freq):
    # This function generates QDs, attached to the Bx port, at a certain 'freq' frequency
    # that should be as a beat note between sidebands. It adds the commands for finesse
    # to track the beat note and  also the _DC QDs used to tune the demodulation phase.
    #
    # The kat file should already contain the Bx_nf/ff_ports attached to appropriate telescopes.

    kat.add_frequency(freq)
    kat.add_frequency(-freq)

    kat.parse(f"""    
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_I {Bx}_nf_port.p1.i f={freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_Q {Bx}_nf_port.p1.i f={freq:.0f} phase=0   pdtype=xsplit
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_I_DC {Bx}_nf_port.p1.i f={freq:.0f} phase=90 
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_Q_DC {Bx}_nf_port.p1.i f={freq:.0f} phase=0  
    
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_I {Bx}_ff_port.p1.i f={freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_Q {Bx}_ff_port.p1.i f={freq:.0f} phase=0   pdtype=xsplit
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_I_DC {Bx}_ff_port.p1.i f={freq:.0f} phase=90 
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_Q_DC {Bx}_ff_port.p1.i f={freq:.0f} phase=0       
    """)


def make_2f_QDs(Bx, kat, freq):
    # This function generates QDs at twice the 'freq' frequency, attached to the Bx port.
    #
    # The kat file should already contain the Bx_nf/ff_ports attached to appropriate telescopes.

    kat.parse(f"""     
    pd1 {Bx}_nf_{2 * freq * 1e-6:.0f}MHz_I {Bx}_nf_port.p1.i f={2 * freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_nf_{2 * freq * 1e-6:.0f}MHz_Q {Bx}_nf_port.p1.i f={2 * freq:.0f} phase=0   pdtype=xsplit
    pd1 {Bx}_ff_{2 * freq * 1e-6:.0f}MHz_I {Bx}_ff_port.p1.i f={2 * freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_ff_{2 * freq * 1e-6:.0f}MHz_Q {Bx}_ff_port.p1.i f={2 * freq:.0f} phase=0   pdtype=xsplit      
    """)


def get_gouy(solution):
    # This function extract the accumulated gouy from the solution of the finesse tracing tool.
    # The beam should be traced from on of the ITMs towards the recycling mirror.

    segments = list(solution.all_segments().items())
    last_key, last_value = segments[-1]
    return last_value[1]["gouy"][-1]

In [ ]:
# create a new Finesse model object,
# this serves as the foundation of the simulation, so we'll name it `base`
base = finesse.Model()

# setup of the interferometer, each line is one component
# ITF version: IMCv3, SB1: 6.37 MHz, SB2: 27.61 MHz, SB3: 78.57 MHz, PRC: [35m, 25°], SRC: [38m, 19°]

PRCL0 = 35.2935
SRCL0 = 38.0083

base.parse_file("AdV_O5_baseline.kat")
base.modes(maxtem=2)

In [ ]:
model_info_virgo.print_model_state(base, verbose=True);

In [ ]:
ns = base.ns

BSHR_to_NIHR = base.sBSsub1.L * ns + base.lBS_NI.L + base.sNIsub.L * ns
BSHR_to_WIHR = base.lBS_WI.L + base.sWIsub.L * ns

model_info.print_sidebands(base)
print("++++ PRC v8p1 (35.2935m, 26° Gouy) 2025/03/13 ++++")
print(f"PRM1 Rc: {base.PRM1.Rc} m")
print(f"PRM2 Rc: {base.PRM2.Rc} m")
print(f"PRM3 Rc: {base.PRM3.Rc} m\n")
print(f"PRM1 -> PRM2: {base.L1_PRC.L}")
print(f"PRM2 -> PRM3: {base.L2_PRC.L}")
print(f"PRM3 -> BS_HR: {base.L3_PRC.L}")
print(
    f"PRCL: {base.L1_PRC.L + base.L2_PRC.L + base.L3_PRC.L + (BSHR_to_NIHR + BSHR_to_WIHR) / 2} m (should be {PRCL0} m)"
)
print("\n")
print("++++ SRC v9p1 (38.0083m, 18° Gouy) 2025/03/13 ++++")
print(f"SRM1 Rc: {base.SRM1.Rc} m")
print(f"SRM2 Rc: {base.SRM2.Rc} m")
print(f"SRM3 Rc: {base.SRM3.Rc} m\n")
print(f"SRM3 -> BS_HR: {base.L3_SRC.L + ns * base.sBSsub2.L}")
print(f"SRM3 -> SRM2: {base.L2_SRC.L} m")
print(f"SRM2 -> SRM1: {base.L1_SRC.L} m")
print(
    f"SRCL: {base.L1_SRC.L + base.L2_SRC.L + base.L3_SRC.L + ns * base.sBSsub2.L + (BSHR_to_NIHR + BSHR_to_WIHR) / 2} m (should be {SRCL0} m)"
)
print("\n")
print(f"BS_HR -> NI_HR: {BSHR_to_NIHR}")
print(f"BS_HR -> WI_HR: {BSHR_to_WIHR}")
print("\n")

## Trace the beam and check the recycling cavities

In [ ]:
model = base.deepcopy()
beam_param = model.beam_trace()

In [ ]:
prop = finesse.tracing.tools.propagate_beam(
    from_node=model.WITM.p1.o,
    to_node=model.WETM.p1.o,
    direction="x",
)
prop.plot()
print(prop)

### Find Gouy phase at the arm's waist to define near field

In [ ]:
def cav_prop_trace(sol, prop):
    """
    This function takes the solution 'sol' of the beam tracing
    tool and the desired property 'prop' chosen among:

    - 'beamsize'
    - 'gouy'
    - 'curvature'

    and gives that property's evolution along the cavity axis.
    It's the same vector plotted by the propagate_beam() tool.
    """

    return list(sol.all_segments().items())[-1][1][1][prop]

In [ ]:
waist_idx = np.argmin(cav_prop_trace(prop, "beamsize"))
arm_gouy_trace = cav_prop_trace(prop, "gouy")

waist_gouy = arm_gouy_trace[waist_idx]
print(f"Gouy phase at the waist in the West arm: {waist_gouy:.4f} deg")

## Check working point

In [ ]:
model_dofs = base.deepcopy()
model_dofs.parse("""
dof MICH NITM.dofs.z -1 NIAR.dofs.z -1 NETM.dofs.z -1 WITM.dofs.z +1 WIAR.dofs.z +1 WETM.dofs.z +1
dof DARM NETM.dofs.z -1 WETM.dofs.z +1
dof CARM NETM.dofs.z +1 WETM.dofs.z +1
dof PRCL PRM1.dofs.z +1
dof SRCL SRM1.dofs.z -1

# Individual mirror motions; they are needed as a "dof" object is what we need to scan later
dof NE_z NETM.dofs.z +1
dof NI_z NITM.dofs.z +1
dof WE_z WETM.dofs.z +1
dof WI_z WITM.dofs.z +1
""")

model_dofs.parse_file("readout_rf.kat")
model_dofs.parse_file("readout_dc.kat")
model_dofs.parse_file("ad_plane_waves.kat")
model_dofs.parse_file("ad_HOMs.kat")
model_dofs.parse_file("pdh_for_TFs.kat")

model_info_virgo.print_model_state(model_dofs, verbose=True);

## Pretuning

In [ ]:
## Run the model and make this changes in series:
model_tuned = model_dofs.deepcopy()

out1 = model_tuned.run(
    fac.TemporaryParameters(
        fac.Series(
            # Switch off the modulators and misalign SR and PR mirrors:
            fac.Change(
                {
                    "eom1.midx": 0,
                    "eom2.midx": 0,
                    "eom4.midx": 0,
                    "eom3.midx": 0,
                    "SRM1.misaligned": True,
                    "PRM1.misaligned": True,
                }
            ),
            # Maximize arm powers:
            fac.Maximize("B7_DC", "NE_z.DC", bounds=[-30, 30], tol=1e-14),
            fac.Maximize("B8_DC", "WE_z.DC", bounds=[-30, 30], tol=1e-14),
            # Minimize dark fringe power:
            fac.Minimize("B1p_DC", "MICH.DC", bounds=[-180, 180], tol=1e-14),
            # Align back PRM:
            fac.Change({"PRM1.misaligned": False}),
            # Maximize PRC power:
            fac.Maximize("B4_CAR_00", "PRCL.DC", bounds=[-180, 180], tol=1e-14),
            # Align back SRM:
            fac.Change({"SRM1.misaligned": False}),
            # Maximize SRC power, then offset by 90 deg:
            fac.Change({"SRCL.DC": 0}),
            fac.Maximize("B1p_CAR_00", "SRCL.DC", bounds=[-180, 180], tol=1e-14),
            fac.Change({"SRCL.DC": -90}, relative=True),
        ),
        # The 'exclude' parameter is needed as TemporaryParameters brings everything back to the starting point at the end of the .run
        exclude=(
            "NE_z.DC",
            "WE_z.DC",
            "MICH.DC",
            "PRCL.DC",
            "SRCL.DC",
        ),
    )
)

In [ ]:
model_info.print_tunings(model_tuned, only_dof=True)
model_info.print_dofs(model_tuned, ["MICH", "PRCL", "SRCL"])

## Check SB fields

In [ ]:
fig, axs = plotting.plot_dof_scans(
    model_tuned,
    (
        ("NE_z", ("B7_DC"), (-4, 4, 500)),
        (
            "PRCL",
            ("B4_USB1_00", "B4_LSB1_00", "B4_USB2_00", "B4_LSB2_00", "B4_DC"),
            (-120, 120, 500),
        ),
        (
            "MICH",
            ("B1p_USB1_00", "B1p_LSB1_00", "B1p_USB2_00", "B1p_LSB2_00", "B1p_CAR_00"),
            (-15, 15, 500),
        ),
        (
            "SRCL",
            (
                "B1p_USB1_00",
                "B1p_LSB1_00",
                "B1p_USB2_00",
                "B1p_LSB2_00",
                "B1p_USB3_00",
                "B1p_LSB3_00",
                "B1p_USB3_10",
                "B1p_LSB3_10",
                "B1p_CAR_00",
            ),
            (-120, 120, 500),
        ),
    ),
    logy=True,
    square_amplitudes=True,
)

plotting.plot_dof_scan(
    model_tuned, "WE_z", ("B8_DC"), (-4, 4, 500), ax=axs[0], square_amplitudes=True
)
axs[0].set_title(
    f"NE_z (DC={(model_tuned.NE_z.DC.eval()):.3g}) (B7)\n"
    f"WE_z (DC={(model_tuned.WE_z.DC.eval()):.3g}) (B8)"
)

## Matching

In [ ]:
plotting.plot_dof_scans(
    model_tuned,
    (
        (
            "PRCL",
            (
                "B4_DC",
                "B4_CAR_00",
                "B4_CAR_20",
                "B4_CAR_02",
                "B4_USB1_00",
                "B4_LSB1_00",
                "B4_USB1_20",
                "B4_LSB1_02",
                "B4_USB2_00",
                "B4_LSB2_00",
                "B4_USB2_20",
                "B4_LSB2_02",
                "B4_USB4_00",
                "B4_LSB4_00",
            ),
            (-180, 180, 500),
        ),
        (
            "SRCL",
            (
                "B1p_DC",
                "B1p_CAR_00",
                "B1p_CAR_20",
                "B1p_CAR_02",
                "B1p_USB1_00",
                "B1p_LSB1_00",
                "B1p_USB1_20",
                "B1p_LSB1_02",
                "B1p_USB2_00",
                "B1p_LSB2_00",
                "B1p_USB2_20",
                "B1p_LSB2_02",
            ),
            (-180, 180, 500),
        ),
    ),
    logy=True,
    figsize=(10, 5),
);

## Run a simple simulations to find the DC power impinging on each PD for normalization purposes

In [ ]:
model_info_virgo.print_powers(
    model_tuned,
    locations={
        "B7": "B7.p1.i",
        "B8": "B8.p1.i",
        "B2": "B2.p1.i",
        "B4": "B4.p1.i",
        "B1p": "B1p.p1.i",
    },
)

# Make Compass plots

In [ ]:
fig, axs = plotting.plot_sensing_matrix(
    model_tuned,
    dofs=["DARM", "CARM", "MICH", "PRCL", "SRCL"],
    signals=[
        "B2_SB1",
        "B2_SB2",
        "B4_SB1",
        "B4_SB2",
        "B6_SB1",
        "B6_SB2",
        "B1p_SB1",
        "B1p_SB2",
    ],
    frequency=1e-3,
)

In [ ]:
# DC powers are needed only for the power-normalized magnitudes.
out_PDs = model_tuned.run(fac.Noxaxis())

# Collect the plotted values, grouped by DOF.
compass_data = {}

for ax in np.ravel(axs):
    for line in ax.lines:
        signal = ax.get_title()
        dof = line.get_label()

        angle = float(line.get_xdata()[0])
        gain_db = float(line.get_ydata()[0])
        magnitude = 10 ** (gain_db / 20)

        power_name = signal.split("_")[0] + "_DC"
        power = float(np.asarray(out_PDs[power_name]).item())

        compass_data.setdefault(dof, {})[signal] = {
            "db": gain_db,
            "magnitude": magnitude,
            "normalized": magnitude / power,
            "angle_deg": np.degrees(angle),
        }

for dof, data in compass_data.items():
    rows = [
        ["OG [dB]", *[v["db"] for v in data.values()]],
        ["OG [mag]", *[v["magnitude"] for v in data.values()]],
        ["OG norm [mag]", *[v["normalized"] for v in data.values()]],
        ["Phi [deg]", *[v["angle_deg"] for v in data.values()]],
    ]

    print(f"\n{dof}")
    print(
        Table(
            [["Quantity", *data.keys()]]
            + [[row[0], *[f"{value:.5g}" for value in row[1:]]] for row in rows],
            headerrow=True,
            headercolumn=True,
            compact=True,
        )
    )


## Check Transfer Functions

Replace TF calculation from parsing katscript to running FrequencyResponse action.
Avoids needing to deepcopy the model.

In [ ]:
freqs = np.geomspace(1, 1000, 101)
out = model_tuned.run(
    fac.FrequencyResponse(f=freqs, inputs=["DARM"], outputs=["B1p.DC"])
)
# Correction of factor 1/0.2 because we measure here at B1p readout which is behind
# B1p_pickoff beam splitter with R=0.2, instead of measuring before beam splitter.
plt.figure()
plt.loglog(freqs, 1 / 0.2 * np.abs(out.out[:, 0, 0]))
plt.xlabel("f [Hz]")
plt.ylabel("TF [W/deg]")
plt.title("DARM to B1p")
plt.show()

### Shake Degree of Freedom

In [ ]:
freqs = np.geomspace(1e-3, 1000, 101)
out_DARM = model_tuned.run(
    fac.FrequencyResponse(f=freqs, inputs=["DARM"], outputs=["B2_SB1.I"])
)
out_CARM = model_tuned.run(
    fac.FrequencyResponse(f=freqs, inputs=["CARM"], outputs=["B4_SB2.Q"])
)
out_PRCL = model_tuned.run(
    fac.FrequencyResponse(f=freqs, inputs=["PRCL"], outputs=["B2_SB1.I"])
)
out_MICH = model_tuned.run(
    fac.FrequencyResponse(f=freqs, inputs=["MICH"], outputs=["B4_SB2.I"])
)
out_SRCL = model_tuned.run(
    fac.FrequencyResponse(f=freqs, inputs=["SRCL"], outputs=["B4_SB2.Q"])
)
fig, ax = plt.subplots(5, 1, figsize=(8, 12))
ax[0].loglog(freqs, np.abs(out_DARM.out[:, 0, 0]))
ax[1].loglog(freqs, np.abs(out_CARM.out[:, 0, 0]))
ax[2].loglog(freqs, np.abs(out_PRCL.out[:, 0, 0]))
ax[3].loglog(freqs, np.abs(out_MICH.out[:, 0, 0]))
ax[4].loglog(freqs, np.abs(out_SRCL.out[:, 0, 0]))
ax[4].set_xlabel("f [Hz]")
ax[0].set_ylabel("DARM to B2_SB1_I")
ax[1].set_ylabel("CARM to B4_SB2_Q")
ax[2].set_ylabel("PRCL to B2_SB1_I")
ax[3].set_ylabel("MICH to B4_SB2_I")
ax[4].set_ylabel("SRCL to B4_SB2_Q")
plt.show()

# We start building the longitudinal sensing matrix

### Check PDH signals before optimizing demodulation phases

In [ ]:
fig, axs = plotting.plot_error_signals(
    model_tuned,
    (
        ("CARM", "B4_SB1", (-0.25, 0.25, 100)),
        ("DARM", "B1p_SB2", (-10, 10, 100)),
        ("MICH", "B4_SB2", (-10, 10, 100)),
        ("PRCL", "B2_SB1", (-10, 10, 100)),
        ("SRCL", "B4_SB2", (-10, 10, 100)),
    ),
)

## Let us prepare the final block of sensors with the correct demodulation phases

In [ ]:
matrix_kat = model_tuned.deepcopy()

# The new compass uses I horizontally and Q vertically, so its angle is
# the phase increment that aligns the response with the I output.
demod_pairs = [
    ("PRCL", "B2_SB1"),
    ("MICH", "B4_SB2"),
    ("DARM", "B1p_SB2"),
    ("CARM", "B4_SB1"),
]
demod_phases = {}
demod_rows = []

for dof, signal in demod_pairs:
    angle_deg = compass_data[dof][signal]["angle_deg"]
    reference_phase = float(model_tuned.get(signal).phase.eval())
    phase_deg = reference_phase + angle_deg
    demod_phases[signal] = phase_deg
    matrix_kat.get(signal).phase = phase_deg
    demod_rows.append([dof, signal, angle_deg, phase_deg])

print(
    Table(
        [["DOF", "Readout", "Compass angle [deg]", "Demod phase [deg]"]]
        + [
            [dof, signal, f"{angle:.5f}", f"{phase:.5f}"]
            for dof, signal, angle, phase in demod_rows
        ],
        headerrow=True,
        headercolumn=True,
        compact=True,
    )
)


### Or optimize demodulation phases directly with Finesse OptimiseRFReadoutPhaseDC action
This gives the exact same phases

In [ ]:
model_optimised_phases = model_tuned.deepcopy()
sol = model_optimised_phases.run(
    fac.OptimiseRFReadoutPhaseDC(
        "DARM",
        "B1p_SB2_I",
        "CARM",
        "B4_SB1_I",
        "PRCL",
        "B2_SB1_I",
        "MICH",
        "B4_SB2_I",
        verbose=True,
    )
)

### Check PDH Signals after optimizing demodulation phases

In [ ]:
fig, axs = plotting.plot_error_signals(
    matrix_kat,
    (
        ("CARM", "B4_SB1", (-0.25, 0.25, 1000)),
        ("DARM", "B1p_SB2", (-10, 10, 1000)),
        ("MICH", "B4_SB2", (-10, 10, 1000)),
        ("PRCL", "B2_SB1", (-10, 10, 1000)),
        ("SRCL", "B4_SB2", (-10, 10, 1000)),
    ),
)

In [ ]:
# Optical gain: signed local slope of each optimized RF error signal.
# SensingMatrixDC returns I + 1j*Q derivatives with respect to DOF.DC,
# in W/degree for these longitudinal degrees of freedom.
og_pairs = [
    ("DARM", "B1p_SB2", "I"),
    ("CARM", "B4_SB1", "I"),
    ("MICH", "B4_SB2", "I"),
    ("PRCL", "B2_SB1", "I"),
    ("SRCL", "B4_SB2", "Q"),
]
og_dofs = [dof for dof, _, _ in og_pairs]
og_readouts = list(dict.fromkeys(signal for _, signal, _ in og_pairs))
og_solution = matrix_kat.run(fac.SensingMatrixDC(og_dofs, og_readouts, d_dof=1e-5))
optical_gains = {}
for i, (dof, signal, quadrature) in enumerate(og_pairs):
    response = og_solution.out[i, og_readouts.index(signal)]
    optical_gains[dof] = float(response.real if quadrature == "I" else response.imag)

# Retain the original variable names, in W/degree.
OG_d, OG_c, OG_m, OG_p, OG_s = [optical_gains[dof] for dof in og_dofs]


In [ ]:
# Finesse mirror tunings use 180 degrees per wavelength.
og_rows = []
for dof, signal, quadrature in og_pairs:
    gain_deg = optical_gains[dof]
    gain_m = gain_deg * 180 / matrix_kat.lambda0
    lock_gain = -1 / gain_deg
    og_rows.append(
        [
            dof,
            f"{signal}_{quadrature}",
            f"{gain_deg:.5g}",
            f"{gain_m:.5g}",
            f"{lock_gain:.5g}",
        ]
    )


gain_table = Table(
    [["DOF", "Error signal", "OG [W/deg]", "OG [W/m]", "Lock gain"]] + og_rows,
    headerrow=True,
    headercolumn=True,
    compact=True,
)
print(gain_table)


## Sensing Matrix: check TFs
### These plots are for unoptimized readout phases in both notebooks

In [ ]:
dofs = ["DARM", "CARM", "MICH", "PRCL", "SRCL"]
readouts = ["B1p_SB2", "B4_SB1", "B4_SB2", "B2_SB1", "B4_SB2"]
readout_quadratures = ["B1p_SB2.I", "B4_SB1.I", "B4_SB2.I", "B2_SB1.I", "B4_SB2.Q"]
freqs = np.geomspace(1e-3, 1e5, 1001)

gains = []

fig, ax = plt.subplots(5, 1, figsize=(8, 10))
for i, (dof, readout) in enumerate(zip(dofs, readouts)):
    out = model_tuned.run(
        fac.FrequencyResponse(
            f=freqs, inputs=[dof], outputs=[f"{readout}.I", f"{readout}.Q"]
        )
    )
    gains.append(out[readout_quadratures[i], dof][0])
    ax[i].loglog(freqs, abs(out[f"{readout}.I", dof]), label=f"{readout}_I")
    ax[i].loglog(freqs, abs(out[f"{readout}.Q", dof]), label=f"{readout}_Q")
    ax[i].set_ylabel(dof, fontsize=20)
    ax[i].legend()
plt.show()

## Sensing Matrix: computation

In [ ]:
import importlib

importlib.reload(model_info)

powers_out = model_info.get_powers(
    model_tuned,
    locations={
        "B1p_SB2": "B1p_SB2.p1.i",
        "B1p_SB22": "B1p_SB2.p1.i",
        "B4_SB1": "B4_SB1.p1.i",
        "B4_SB12": "B4_SB1.p1.i",
        "B2_SB1": "B2_SB1.p1.i",
        "B2_SB12": "B2_SB1.p1.i",
        "B4_SB2": "B4_SB2.p1.i",
        "B4_SB22": "B4_SB2.p1.i",
    },
)
powers = [1 / val["total"] for val in powers_out.values()]

print(powers)
scale = [powers, powers, powers, powers, powers]

model_info.print_sensing_matrix(
    model_tuned,
    dofs=["DARM", "CARM", "MICH", "PRCL", "SRCL"],
    readouts=["B1p_SB2", "B4_SB1", "B2_SB1", "B4_SB2"],
    scale=scale,
    frequency=1e-3,
)

In [ ]:
from finesse.utilities.tables import NumberTable

solution = model_tuned.run(
    fac.SensingMatrixDC(
        dofs=["DARM", "CARM", "MICH", "PRCL", "SRCL"],
        readouts=["B1p_SB2", "B4_SB1", "B2_SB1", "B4_SB2"],
    )
)
matrix, dof_names, readout_names = solution.matrix_data()
powers = [1 / val["total"] * 360 / model_tuned.lambda0 for val in powers_out.values()]

print(
    NumberTable(
        matrix * np.asarray(powers)[None, :],
        rownames=dof_names,
        colnames=readout_names,
        numfmt="{:.8G}",
    )
)


## Test the combination of TMs tilt

In [ ]:
test_kat = matrix_kat.deepcopy()
test_kat.parse("""
mathd NE NETM.xbeta
mathd NI NITM.xbeta
mathd WE WETM.xbeta
mathd WI WITM.xbeta
""")

In [ ]:
out = test_kat.run(fac.Xaxis("DIFFm_set", "lin", -1e-7, 1e-7, 300))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(out.x1, out["NE"], label="NE")
ax.plot(out.x1, out["NI"], label="NI")
ax.plot(out.x1, out["WE"], label="WE")
ax.plot(out.x1, out["WI"], label="WI")
ax.legend()
ax.grid()
ax.set_xlabel("var [rad]")
ax.set_ylabel("mir [rad]")
fig.set_tight_layout(True)

In [ ]:
tele = f"""
s sB2_to_bench B2_pickoff.p4 B2_split.p1 L=0
bs B2_split R=0.5 T=0.5 alpha=45
s B2_nf_tele B2_split.p2 B2_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B2_ff_tele B2_split.p3 B2_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B2_nf_port
nothing B2_ff_port

s sB4_to_bench PRM2.p3 B4_split.p1 L=0
bs B4_split R=0.5 T=0.5 alpha=45
s B4_nf_tele B4_split.p2 B4_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B4_ff_tele B4_split.p3 B4_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B4_nf_port
nothing B4_ff_port

s sB1p_to_bench B1p_pickoff.p2 B1p_split.p1 L=0
bs B1p_split R=0.5 T=0.5 alpha=45
s B1p_nf_tele B1p_split.p2 B1p_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B1p_ff_tele B1p_split.p3 B1p_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B1p_nf_port
nothing B1p_ff_port

s sB7_to_bench NETM.p2 B7_split.p1 L=0
bs B7_split R=0.5 T=0.5 alpha=45
s B7_nf_tele B7_split.p2 B7_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B7_ff_tele B7_split.p3 B7_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B7_nf_port
nothing B7_ff_port

s sB8_to_bench WETM.p2 B8_split.p1 L=0
bs B8_split R=0.5 T=0.5 alpha=45
s B8_nf_tele B8_split.p2 B8_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B8_ff_tele B8_split.p3 B8_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B8_nf_port
nothing B8_ff_port
"""


## Sweep all angular dofs
Get lock gains from gain table

In [ ]:
lock_kat = matrix_kat.deepcopy()
lock_kat.modes(maxtem=4)
lock_kat.parse(
    """
lock DARM_lock B1p_SB2_I  DARM.DC  1  5e-14  enabled=True offset=0
lock CARM_lock B4_SB1_I   CARM.DC  1  5e-14  enabled=True offset=0
lock MICH_lock B4_SB2_I   MICH.DC  1  1e-12  enabled=True offset=0
lock PRCL_lock B2_SB1_I   PRCL.DC  1  1e-12  enabled=True offset=0
lock SRCL_lock B4_SB2_Q   SRCL.DC  1  1e-10  enabled=True offset=0
"""
)

In [ ]:
lock_kat.parse(tele)
lock_kat.parse_file("QDs_DC.kat")
lock_kat.parse_file("QDs_RF.kat")

make_SoS_QDs("B1p", lock_kat, lock_kat.eom3.f - lock_kat.eom2.f)
make_SoS_QDs("B1p", lock_kat, lock_kat.eom2.f - lock_kat.eom1.f)

lock_kat.add_frequency(2 * lock_kat.eom1.f)
lock_kat.add_frequency(-2 * lock_kat.eom1.f)
lock_kat.add_frequency(2 * lock_kat.eom2.f)
lock_kat.add_frequency(-2 * lock_kat.eom2.f)

make_2f_QDs("B2", lock_kat, lock_kat.eom1.f.value)
make_2f_QDs("B2", lock_kat, lock_kat.eom2.f.value)
make_2f_QDs("B4", lock_kat, lock_kat.eom1.f.value)
make_2f_QDs("B4", lock_kat, lock_kat.eom2.f.value)
make_2f_QDs("B1p", lock_kat, lock_kat.eom1.f.value)
make_2f_QDs("B1p", lock_kat, lock_kat.eom2.f.value)

In [ ]:
out = {}
AA_dofs = [
    "PRM1",
    "PRM2",
    "PRM3",
    "SRM1",
    "SRM2",
    "SRM3",
    "BS",
    "IBshift",
    "IBtilt",
    "DIFFp",
    "DIFFm",
    "COMMp",
    "COMMm",
]

In [ ]:
lock_kat.run(fac.SetLockGains())
model_info.print_lock_status(lock_kat)

### Try just running the first lock
In the original notebooks gains were calculated incorrectly. With correct gain
calculation above, locks also run with method='proportional' if a small scale factor is
used to prevent feedback from overshooting.

In [ ]:
locktest_kat = lock_kat.deepcopy()
locktest_kat.run(fac.RunLocks(method="newton", display_progress=True))

locktest_kat = lock_kat.deepcopy()
locktest_kat.run(
    fac.RunLocks(method="proportional", scale_factor=0.3, display_progress=True)
)

# Full calculation

In [ ]:
for dof in AA_dofs:
    print(f"Now sweeping {dof}...", end="\r")
    time_start = time.time()

    out[dof] = lock_kat.run(
        f"xaxis({dof + '_set'}, lin, 0, 1e-12, 3, pre_step=RunLocks(method='proportional', scale_factor=0.2, show_progress_bar=True))"
    )

    time_stop = time.time()
    elapsed_time = time_stop - time_start
    print(f"Done sweeping {dof}. Elapsed time: {elapsed_time:.2f} seconds.")
print("All done")